# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show all available record sets by their @id and name
all_record_sets = list(dataset.record_sets.values())
print(f"Found {len(all_record_sets)} record sets:")
for rs in all_record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    # List the fields for each record set
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets.values()]
print(f"RecordSet @ids: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Shape of record set {record_set_id}: {df.shape}")
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Columns for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll attempt on the first available record set
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    print(f"Sample rows for EDA from {record_set_id}:")
    display(df.head())

    # Look for a numeric field (try to select the first float/integer-type column by inspecting the dtypes)
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field '{numeric_field}' for EDA.")
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (75th percentile):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field (prefer a string/categorical)
        other_fields = [col for col in df.columns if col != numeric_field]
        possible_group_fields = df[other_fields].select_dtypes(include=['object']).columns.tolist()
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    # Plot the distribution of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If both a group and numeric field are available
    if possible_group_fields:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded the dataset via its Croissant schema, explored available record sets, extracted records into dataframes, performed basic EDA including filtering and normalization, and visualized the distributions of selected numeric attributes. For more advanced/targeted analysis, review the data dictionary and the Croissant schema to utilize domain-specific fields and structures.*